# Getting started with Ontospy

This notebook shows the basics of [Ontospy](https://github.com/lambdamusic/ontospy):

1. loading an ontology
2. printing its class and property hierarchies
3. inspecting classes, properties and individuals
4. turning the results into [pandas](https://pandas.pydata.org/) data frames

It uses the Pizza ontology, which comes with Ontospy's test files. To run it:

```
$ pip install ontospy pandas jupyter
```

In [1]:
import ontospy
import pandas as pd

print(ontospy.VERSION)

v2.3


## 1. Load an ontology

`Ontospy()` takes a file path, a URL, or a list of them. It reads the RDF, then extracts the ontologies, classes, properties etc. it finds.

`hide_individuals=False` also extracts the individuals (instances of classes), which are skipped by default.

In [2]:
g = ontospy.Ontospy("../ontospy/tests/rdf/pizza.ttl", hide_individuals=False, verbose=True)

Reading: <../ontospy/tests/rdf/pizza.ttl>


.. trying rdf serialization: <turtle>


..... success!


----------
Loaded 2332 triples.
----------


RDF sources loaded successfully: 1 of 1.


...

..object...........: 8


Concepts (SKOS)....: 0


Shapes (SHACL).....: 0


Individuals........: 5


----------


Other ways to load data:

```python
g = ontospy.Ontospy("http://xmlns.com/foaf/spec/index.rdf")           # from a URL
g = ontospy.Ontospy(["file1.ttl", "file2.ttl"])                       # several sources
g = ontospy.Ontospy(sparql_endpoint="https://dbpedia.org/sparql")     # a SPARQL endpoint
```

In [3]:
g.stats()

[('Ontologies', 1),
 ('Triples', 2332),
 ('Classes', 99),
 ('Properties', 8),
 ('Annotation Properties', 0),
 ('Object Properties', 8),
 ('Datatype Properties', 0),
 ('Skos Concepts', 0),
 ('Data Shapes', 0),
 ('Individuals', 5),
 ('Data Sources', 1)]

## 2. Print the hierarchies

`printClassTree()` prints the whole class hierarchy. Here it's limited to one branch, `pizza:NamedPizza`, as the full tree is long.

In [4]:
named_pizza = g.get_class(uri="http://www.co-ode.org/ontologies/pizza/pizza.owl#NamedPizza")
g.printClassTree(named_pizza)

pizza.owl:NamedPizza


----pizza.owl:American


----pizza.owl:AmericanHot


----pizza.owl:Cajun


----pizza.owl:Capricciosa


...

----pizza.owl:Siciliana


----pizza.owl:SloppyGiuseppe


----pizza.owl:Soho


----pizza.owl:UnclosedPizza


----pizza.owl:Veneziana


In [5]:
g.printPropertyTree()

pizza.owl:hasCountryOfOrigin


pizza.owl:hasIngredient


----pizza.owl:hasBase


----pizza.owl:hasTopping


pizza.owl:hasSpiciness


pizza.owl:isIngredientOf


----pizza.owl:isBaseOf


----pizza.owl:isToppingOf


## 3. Inspect entities

Everything Ontospy extracts is available as Python objects:

- `g.all_ontologies`, `g.all_classes`, `g.all_properties`, `g.all_individuals`, `g.all_skos_concepts`, `g.all_shapes`
- `g.get_class()`, `g.get_property()`, `g.get_individual()` find an entity by `uri`, by position (`id`), or by matching part of its name (`match`)

In [6]:
g.get_class(match="spicy")

[<Class *http://www.co-ode.org/ontologies/pizza/pizza.owl#SpicyPizza*>,
 <Class *http://www.co-ode.org/ontologies/pizza/pizza.owl#SpicyPizzaEquivalent*>,
 <Class *http://www.co-ode.org/ontologies/pizza/pizza.owl#SpicyTopping*>]

In [7]:
american = g.get_class(match="American")[0]
american.describe()

http://www.co-ode.org/ontologies/pizza/pizza.owl#American


----------------


Parents......: 1


Children.....: 0


Ancestors....: 4


Descendants..: 0


Domain of....: 0


Range of.....: 0


Instances....: 0


----------------


In [8]:
print("Parents:    ", named_pizza.parents())
print("Children:   ", len(named_pizza.children()))
print("Ancestors:  ", named_pizza.ancestors())
print("Label:      ", named_pizza.bestLabel())
print("Description:", named_pizza.bestDescription())

Parents:     [<Class *http://www.co-ode.org/ontologies/pizza/pizza.owl#Pizza*>]
Children:    23
Ancestors:   [<Class *http://www.co-ode.org/ontologies/pizza/pizza.owl#Pizza*>, <Class *http://www.co-ode.org/ontologies/pizza/pizza.owl#Food*>, <Class *http://www.co-ode.org/ontologies/pizza/pizza.owl#DomainConcept*>]
Label:       PizzaComUmNome
Description: A pizza that can be found on a pizza menu


Properties know their domains and ranges:

In [9]:
topping = g.get_property(match="hasTopping")[0]
print(topping.rdftype_qname)
print("Domains:", topping.domains)
print("Ranges: ", topping.ranges)

owl:ObjectProperty
Domains: [<Class *http://www.co-ode.org/ontologies/pizza/pizza.owl#Pizza*>]
Ranges:  [<Class *http://www.co-ode.org/ontologies/pizza/pizza.owl#PizzaTopping*>]


Individuals know which classes they're instances of:

In [10]:
for x in g.all_individuals:
    print(x.qname, "is a", [c.qname for c in x.instance_of()])

pizza.owl:America is a ['pizza.owl:Country']
pizza.owl:England is a ['pizza.owl:Country']
pizza.owl:France is a ['pizza.owl:Country']
pizza.owl:Germany is a ['pizza.owl:Country']
pizza.owl:Italy is a ['pizza.owl:Country']


Each entity also keeps the RDF that defines it:

In [11]:
print(american.rdf_source())

@prefix : <http://www.co-ode.org/ontologies/pizza/pizza.owl#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

:American a owl:Class ;
    rdfs:label "Americana"@pt ;
    rdfs:subClassOf [ a owl:Restriction ;
            owl:allValuesFrom [ a owl:Class ;
                    owl:unionOf ( :MozzarellaTopping :PeperoniSausageTopping :TomatoTopping ) ] ;
            owl:onProperty :hasTopping ],
        [ a owl:Restriction ;
            owl:hasValue :America ;
            owl:onProperty :hasCountryOfOrigin ],
        [ a owl:Restriction ;
            owl:onProperty :hasTopping ;
            owl:someValuesFrom :MozzarellaTopping ],
        [ a owl:Restriction ;
            owl:onProperty :hasTopping ;
            owl:someValuesFrom :PeperoniSausageTopping ],
        [ a owl:Restriction ;
            owl:onProperty :hasTopping ;
            owl:someValuesFrom :TomatoTopping 

## 4. Data frames

Ontospy's objects are easy to turn into tables. The summary statistics:

In [12]:
pd.DataFrame(g.stats(), columns=["Entity type", "Count"])

,Entity type,Count
0,Ontologies,1
1,Triples,2332
2,Classes,99
3,Properties,8
4,Annotation Properties,0
5,Object Properties,8
6,Datatype Properties,0
7,Skos Concepts,0
8,Data Shapes,0
9,Individuals,5


All classes, with their parents and number of subclasses:

In [13]:
classes = pd.DataFrame([
    {
        "qname": c.qname,
        "label": c.bestLabel(),
        "parents": ", ".join(p.qname for p in c.parents()),
        "subclasses": len(c.children()),
        "uri": str(c.uri),
    }
    for c in g.all_classes
])
print(len(classes), "classes")
classes.head(10)

99 classes


,qname,label,parents,subclasses,uri
0,pizza.owl:American,Americana,pizza.owl:NamedPizza,0,http://www.co-ode.org/ontologies/pizza/pizza.o...
1,pizza.owl:AmericanHot,AmericanaPicante,pizza.owl:NamedPizza,0,http://www.co-ode.org/ontologies/pizza/pizza.o...
2,pizza.owl:AnchoviesTopping,CoberturaDeAnchovies,pizza.owl:FishTopping,0,http://www.co-ode.org/ontologies/pizza/pizza.o...
3,pizza.owl:ArtichokeTopping,CoberturaDeArtichoke,pizza.owl:VegetableTopping,0,http://www.co-ode.org/ontologies/pizza/pizza.o...
4,pizza.owl:AsparagusTopping,CoberturaDeAspargos,pizza.owl:VegetableTopping,0,http://www.co-ode.org/ontologies/pizza/pizza.o...
5,pizza.owl:Cajun,Cajun,pizza.owl:NamedPizza,0,http://www.co-ode.org/ontologies/pizza/pizza.o...
6,pizza.owl:CajunSpiceTopping,CoberturaDeCajun,pizza.owl:HerbSpiceTopping,0,http://www.co-ode.org/ontologies/pizza/pizza.o...
7,pizza.owl:CaperTopping,CoberturaDeCaper,pizza.owl:VegetableTopping,0,http://www.co-ode.org/ontologies/pizza/pizza.o...
8,pizza.owl:Capricciosa,Capricciosa,pizza.owl:NamedPizza,0,http://www.co-ode.org/ontologies/pizza/pizza.o...
9,pizza.owl:Caprina,Caprina,pizza.owl:NamedPizza,0,http://www.co-ode.org/ontologies/pizza/pizza.o...


In [14]:
# eg the classes with the most direct subclasses
classes.sort_values("subclasses", ascending=False).head(5)

,qname,label,parents,subclasses,uri
48,pizza.owl:NamedPizza,PizzaComUmNome,pizza.owl:Pizza,23,http://www.co-ode.org/ontologies/pizza/pizza.o...
93,pizza.owl:VegetableTopping,CoberturaDeVegetais,pizza.owl:PizzaTopping,14,http://www.co-ode.org/ontologies/pizza/pizza.o...
64,pizza.owl:PizzaTopping,CoberturaDaPizza,pizza.owl:Food,8,http://www.co-ode.org/ontologies/pizza/pizza.o...
10,pizza.owl:CheeseTopping,CoberturaDeQueijo,pizza.owl:PizzaTopping,6,http://www.co-ode.org/ontologies/pizza/pizza.o...
40,pizza.owl:MeatTopping,CoberturaDeCarne,pizza.owl:PizzaTopping,4,http://www.co-ode.org/ontologies/pizza/pizza.o...


All properties, with their type, domains and ranges:

In [15]:
properties = pd.DataFrame([
    {
        "qname": p.qname,
        "type": p.rdftype_qname,
        "domains": ", ".join(d.qname for d in p.domains),
        "ranges": ", ".join(r.qname for r in p.ranges),
    }
    for p in g.all_properties
])
properties

,qname,type,domains,ranges
0,pizza.owl:hasBase,owl:ObjectProperty,pizza.owl:Pizza,pizza.owl:PizzaBase
1,pizza.owl:hasCountryOfOrigin,owl:ObjectProperty,,
2,pizza.owl:hasIngredient,owl:ObjectProperty,pizza.owl:Food,pizza.owl:Food
3,pizza.owl:hasSpiciness,owl:ObjectProperty,,pizza.owl:Spiciness
4,pizza.owl:hasTopping,owl:ObjectProperty,pizza.owl:Pizza,pizza.owl:PizzaTopping
5,pizza.owl:isBaseOf,owl:ObjectProperty,pizza.owl:PizzaBase,pizza.owl:Pizza
6,pizza.owl:isIngredientOf,owl:ObjectProperty,pizza.owl:Food,pizza.owl:Food
7,pizza.owl:isToppingOf,owl:ObjectProperty,pizza.owl:PizzaTopping,pizza.owl:Pizza


## 5. SPARQL queries

`query()` runs a SPARQL query on the graph. By default it returns a list of result rows; `format="df"` returns a data frame instead (it needs pandas). Other formats: `csv`, `json`, `xml`, `txt`.

The Pizza ontology's labels are in Portuguese, so this query asks for those.

In [16]:
q = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX pizza: <http://www.co-ode.org/ontologies/pizza/pizza.owl#>

SELECT ?pizza ?label
WHERE {
    ?pizza rdfs:subClassOf pizza:NamedPizza ;
           rdfs:label ?label .
    FILTER(lang(?label) = "pt")
}
ORDER BY ?label
"""
g.query(q, format="df")

,pizza,label
0,http://www.co-ode.org/ontologies/pizza/pizza.o...,Americana
1,http://www.co-ode.org/ontologies/pizza/pizza.o...,AmericanaPicante
2,http://www.co-ode.org/ontologies/pizza/pizza.o...,Cajun
3,http://www.co-ode.org/ontologies/pizza/pizza.o...,Capricciosa
4,http://www.co-ode.org/ontologies/pizza/pizza.o...,Caprina
5,http://www.co-ode.org/ontologies/pizza/pizza.o...,CoberturaPrinceCarlo
6,http://www.co-ode.org/ontologies/pizza/pizza.o...,Cogumelo
7,http://www.co-ode.org/ontologies/pizza/pizza.o...,Fiorentina
8,http://www.co-ode.org/ontologies/pizza/pizza.o...,FrutosDoMar
9,http://www.co-ode.org/ontologies/pizza/pizza.o...,Giardiniera


In [17]:
print(g.query(q, format="csv")[:300])

pizza,label
http://www.co-ode.org/ontologies/pizza/pizza.owl#American,Americana
http://www.co-ode.org/ontologies/pizza/pizza.owl#AmericanHot,AmericanaPicante
http://www.co-ode.org/ontologies/pizza/pizza.owl#Cajun,Cajun
http://www.co-ode.org/ontologies/pizza/pizza.owl#Capricciosa,Capricciosa
htt


## Next steps

- `ontospy gendocs` generates HTML or markdown documentation for an ontology, see the [documentation](http://lambdamusic.github.io/Ontospy/)
- `ontospy shell` explores an ontology interactively